In [1]:

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from dataclasses import dataclass
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from datasets import load_dataset, concatenate_datasets, ClassLabel
import torch 

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

import warnings 
warnings.filterwarnings('ignore')

c:\Users\LM23-2\AppData\Local\anaconda3\envs\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LM23-2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LM23-2\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LM23-2\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
MISTRAL = "mistralai/Mistral-7B-Instruct-v0.3"
BART_MNLI = "facebook/bart-large-mnli"

DATASETS_LINKS: list[str] = [
    'AdamCodd/emotion-balanced',
    'dair-ai/emotion',
    # 'philschmid/emotion',
    'SetFit/emotion',
    'mteb/emotion'
]

DATASET_MAPPINGS = [
    {
        0: "sadness",
        1: "joy",
        2: "love",
        3: "anger",
        4: "fear",
        5: "surprise"
    },
    {
        0: "sadness",
        1: "joy",
        2: "love",
        3: "anger",
        4: "fear",
        5: "surprise"
    },
    {
        0: "sadness",
        1: "joy",
        2: "love",
        3: "anger",
        4: "fear",
        5: "surprise"
    },
    {
        0: "sadness",
        1: "joy",
        2: "love",
        3: "anger",
        4: "fear",
        5: "surprise"
    },
] 

In [3]:
@dataclass
class DataLoaderSettings:
    dataset_link: str
    keys: list[str]
    text_col: str
    label_col: str
    mappings: dict

@dataclass
class TrainingSettings:
    model_name: str

In [4]:

class DataLoader:
    def __init__(self, loader_settings: DataLoaderSettings):
        self.stop_words = stopwords.words('english')
        self.settings = loader_settings
        self.loaded = self.load_dataset()
        self.processed = self.process()
        self.convert_labels_to_classlabel()

        print(f'Loaded: {self.loaded}')
        print(f'Processed: {self.processed}')

    def load_dataset(self) -> bool:
        try:
            dataset = load_dataset(self.settings.dataset_link, trust_remote_code=True)
            merged_dataset = None

            if not isinstance(dataset, dict):
                self.dataset = dataset
                return True

            for key in self.settings.keys:
                if key not in dataset:
                    continue

                dataset_partition = dataset[key]

                if merged_dataset == None:
                    merged_dataset = dataset_partition
                else:
                    merged_dataset = concatenate_datasets([merged_dataset, dataset_partition])

            self.dataset = merged_dataset
            return True
        except Exception as e:
            print(f'Something went wrong when trying to load dataset from link {self.settings.dataset_link}')
            print(f'Got error {e}')
            return False

    def convert_labels_to_classlabel(self):
        unique_labels = list(set(self.dataset[self.settings.label_col]))
        class_label_feature = ClassLabel(num_classes=len(unique_labels), names=[str(label) for label in unique_labels])

        self.dataset = self.dataset.map(lambda example: {self.settings.label_col: class_label_feature.str2int(str(example[self.settings.label_col]))})
        self.dataset = self.dataset.cast_column(self.settings.label_col, class_label_feature)
        print(type(self.dataset[0]['label']))
        
    def get_data_each_label(self, data_per_label=1, random_seed=42) -> dict:
        '''
        Gets Data Per Label in the dataset that has been loaded with a random seed.
        
        Args:
        - data_per_label: The amount of text and label pairing that is going to be retrieved per label.
        - random_seed: The random seed that is going to be used for selecting the data.
        
        Returns:
        - A dictionary where keys are labels and values are lists of sampled text data.
        '''
        if not self.loaded:
            raise ValueError("Dataset has not been loaded")

        label_col = self.settings.label_col
        text_col = self.settings.text_col

        data_by_label = {}
        dataset_df = self.dataset.to_pandas()

        unique_labels = dataset_df[label_col].unique()

        for label in unique_labels:
            subset = dataset_df[dataset_df[label_col] == label]
            sampled_data = subset.sample(n=min(data_per_label, len(subset)), random_state=random_seed)
            data_by_label[label] = sampled_data[text_col].tolist()

        return data_by_label
    
    def map_label(self, label: int) -> str:
        return self.settings.mappings[label]

    def process(self) -> bool:
        if not self.loaded:
            raise ValueError("Dataset has not been loaded")

        def process_text(sample) -> str:
            text = sample[self.settings.text_col]
            words = self.tokenize(text)
            words = self.remove_stopwords(words)

            sample[self.settings.text_col] = ' '.join(words)
            return sample

        try:
            self.dataset = self.dataset.map(process_text)
            return True
        except Exception as e:
            print(e)
            return False

    def tokenize(self, text) -> list[str]:
        words = word_tokenize(text)
        return words

    def remove_stopwords(self, words) -> list[str]:
        words = [word for word in words if word not in self.stop_words and word.isalpha()]
        return words

In [5]:
settings = DataLoaderSettings(
    dataset_link=DATASETS_LINKS[0],
    keys=['train', 'test', 'validation'],
    label_col='label',
    text_col='text',
    mappings=DATASET_MAPPINGS[0]
)

loader = DataLoader(loader_settings=settings)

<class 'int'>
Loaded: True
Processed: True


In [6]:
prompting_datas = loader.get_data_each_label()

for key in prompting_datas:
    texts = prompting_datas[key]
    label = loader.map_label(key)
    print(f'Label: {label}')
    print(texts)

Label: anger
['found counting minutes feeling agitated']
Label: love
['really like anna faris character feels genderless fact gets behave like slutty sloppy guys rom coms really judged end']
Label: fear
['feel strange note even changes preferences self image still partial people']
Label: joy
['feel smart sometimes']
Label: sadness
['explained feel disappointed forgot past deliverance really cared even slightest detail']
Label: surprise
['feel overwhelmed frustrated tired taken granted advantage nobody blame makes frustrated']


In [7]:
def generate_few_shot_prompt(prompting_datas):
    """
    Generates a few-shot prompt for in-context learning.

    Args:
    - prompting_datas: Dictionary where keys are labels and values are lists of sample texts.

    Returns:
    - A formatted string that can be used as a prompt for an LLM.
    """
    prompt = "Below are examples of text classifications. Given a text, classify it into one of the categories.\n\n"

    for key, texts in prompting_datas.items():
        label = loader.map_label(key)
        for text in texts:
            prompt += f"Text: {text}\nLabel: {label}\n\n"

    prompt += "Now, classify the following text:\nText: "
    return prompt

In [8]:
few_shot_prompt = generate_few_shot_prompt(prompting_datas=prompting_datas)

new_text = "I love you so much\nLabel:\n"
full_prompt = few_shot_prompt + new_text

print(full_prompt)

Below are examples of text classifications. Given a text, classify it into one of the categories.

Text: found counting minutes feeling agitated
Label: anger

Text: really like anna faris character feels genderless fact gets behave like slutty sloppy guys rom coms really judged end
Label: love

Text: feel strange note even changes preferences self image still partial people
Label: fear

Text: feel smart sometimes
Label: joy

Text: explained feel disappointed forgot past deliverance really cared even slightest detail
Label: sadness

Text: feel overwhelmed frustrated tired taken granted advantage nobody blame makes frustrated
Label: surprise

Now, classify the following text:
Text: I love you so much
Label:



In [14]:
tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2', trust_remote_code=True)

In [21]:
input_ids = tokenizer("What is the emotion from this text\n im really sad", return_tensors="pt").input_ids.to(model.device)

with torch.no_grad():
    output = model.generate(
        input_ids, 
        max_new_tokens=20,  # Limit output length
        temperature=0.5,  # Adjust temperature for better control
        eos_token_id=tokenizer.eos_token_id
    )

# Decode and extract only the label
predicted_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(predicted_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is the emotion from this text
 im really sad

I'm so sorry

I'm so sorry

I'm so sorry




#### Zero Shot Learning

In [42]:
def zero_shot_learning(settings: TrainingSettings, label_mapping: dict, prompting_datas: dict):
    classifier = pipeline(task='zero-shot-classification', model=settings.model_name)
    labels = list(label_mapping.values())
    for label in prompting_datas:
        texts = prompting_datas[label]
        
        for text in texts:
            result = classifier(text, labels)
            print(result)

In [27]:
training_settings = TrainingSettings(
    model_name=BART_MNLI
)

In [43]:
data = loader.get_data_each_label(1)
data

{np.int64(3): ['found counting minutes feeling agitated'],
 np.int64(2): ['really like anna faris character feels genderless fact gets behave like slutty sloppy guys rom coms really judged end'],
 np.int64(4): ['feel strange note even changes preferences self image still partial people'],
 np.int64(1): ['feel smart sometimes'],
 np.int64(0): ['explained feel disappointed forgot past deliverance really cared even slightest detail'],
 np.int64(5): ['feel overwhelmed frustrated tired taken granted advantage nobody blame makes frustrated']}

In [44]:
zero_shot_learning(
    settings=training_settings,
    label_mapping=DATASET_MAPPINGS[0],
    prompting_datas=data
)

Device set to use cpu


{'sequence': 'found counting minutes feeling agitated', 'labels': ['anger', 'surprise', 'fear', 'sadness', 'joy', 'love'], 'scores': [0.8133013844490051, 0.13138757646083832, 0.04147407412528992, 0.005411892663687468, 0.0052625625394284725, 0.003162478329613805]}
{'sequence': 'really like anna faris character feels genderless fact gets behave like slutty sloppy guys rom coms really judged end', 'labels': ['surprise', 'joy', 'anger', 'sadness', 'fear', 'love'], 'scores': [0.48381274938583374, 0.27278977632522583, 0.10332874208688736, 0.05008931830525398, 0.049075763672590256, 0.04090365022420883]}
{'sequence': 'feel strange note even changes preferences self image still partial people', 'labels': ['surprise', 'fear', 'sadness', 'anger', 'joy', 'love'], 'scores': [0.840140163898468, 0.05477932095527649, 0.0449519082903862, 0.02822720818221569, 0.017477326095104218, 0.014424107037484646]}
{'sequence': 'feel smart sometimes', 'labels': ['surprise', 'joy', 'fear', 'love', 'anger', 'sadness'